In [0]:
%sql

-- SCRIPT: 6.4_serie_temporal_e_percentil
-- LOCAL: 3_gold/src/6_monitor_presenca/
-- OBJETIVO: Gerar relatório mensal de engajamento usando como fonte a tabela do script /3_gold/6_monitor_presenca/6.1_analise_absenteismo 
-- ENTREGA: Série temporal de engajamento com queda após eventos críticos (escândalos, reeleição)

CREATE OR REPLACE TABLE gold_relatorio_mensal_engajamento AS
WITH metricas_mensais AS (
    SELECT 
        id_deputado,
        nome_deputado,
        partido,        
        date_trunc('month', CAST(data_voto AS DATE)) as mes_referencia,
        COUNT(id_votacao) as sessoes_mes,
        SUM(CASE WHEN status_presenca = 'PRESENTE' THEN 1 ELSE 0 END) as presencas_mes
    FROM gold_monitor_absenteismo
    GROUP BY 1, 2, 3, 4
),
calculo_score AS (
    -- Calcula a nota do mês
    SELECT 
        *,
        ROUND((presencas_mes / sessoes_mes) * 10, 2) as score_mes
    FROM metricas_mensais
)

-- Calculo do Percentil 
SELECT 
    *,
    ROUND(PERCENT_RANK() OVER (PARTITION BY mes_referencia ORDER BY score_mes) * 100, 1) as percentil_ranking
FROM calculo_score;

-- Visualização para o relatório
SELECT * FROM gold_relatorio_mensal_engajamento 
ORDER BY mes_referencia DESC, score_mes DESC;